# 📊 UserFlow Analytics – Product Growth & A/B Testing Platform

## Business Problem

UserFlow is an e-commerce platform that wants to improve customer engagement, increase conversion rates, and maximize revenue.

The product team recently launched a new website experience (Variant B) while continuing to serve the existing design (Variant A).

As a Data Analyst, the objective is to analyze customer behavior, identify growth opportunities, evaluate the A/B experiment, and provide actionable business recommendations using Python, SQL, statistics, and interactive visualizations.

---

### Project Objectives

- Perform data cleaning
- Conduct Exploratory Data Analysis (EDA)
- Build business KPIs
- Analyze customer behavior
- Perform Funnel Analysis
- Conduct Cohort Analysis
- Perform Retention Analysis
- Evaluate an A/B Test
- Generate Executive Business Insights

In [1]:
!pip install duckdb --quiet

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

import duckdb

from scipy import stats

In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

In [4]:
print("=" * 60)
print("UserFlow Analytics Project")
print("=" * 60)

print("Pandas Version :", pd.__version__)
print("NumPy Version  :", np.__version__)

print("\nEnvironment Ready!")

UserFlow Analytics Project
Pandas Version : 2.2.2
NumPy Version  : 2.0.2

Environment Ready!


In [5]:
import pandas as pd

users = pd.read_csv("users.csv")
products = pd.read_csv("products.csv")
orders = pd.read_csv("orders.csv")
order_items = pd.read_csv("order_items.csv")
events = pd.read_csv("events.csv")
reviews = pd.read_csv("reviews.csv")

print("✅ All datasets loaded successfully!")

✅ All datasets loaded successfully!


In [6]:
datasets = {
    "Users": users,
    "Products": products,
    "Orders": orders,
    "Order Items": order_items,
    "Events": events,
    "Reviews": reviews
}

for name, df in datasets.items():
    print("=" * 70)
    print(name.upper())
    print("=" * 70)
    print("Shape :", df.shape)
    print()
    display(df.head())
    print("\n")

USERS
Shape : (10000, 6)



,user_id,name,email,gender,city,signup_date
0,U000001,Angel Hill,donaldgarcia@example.net,Other,New Roberttown,2025-03-13
1,U000002,Jesse Guzman,jennifermiles@example.com,Male,South Bridget,2024-03-05
2,U000003,Adam Shaffer,jpeterson@example.org,Male,Curtisfurt,2025-07-07
3,U000004,Melanie Munoz,blairamanda@example.com,Other,New Kellystad,2024-03-07
4,U000005,Janet Williams,kendragalloway@example.org,Female,South Joshuastad,2025-01-29




PRODUCTS
Shape : (2000, 6)



,product_id,product_name,category,brand,price,rating
0,P000001,Astra Be,Clothing,Astra,157.89,4.08
1,P000002,NeoTech Someone,Groceries,NeoTech,21.46,3.87
2,P000003,Acme Discuss,Sports,Acme,265.37,3.46
3,P000004,Nimbus South,Electronics,Nimbus,541.41,4.14
4,P000005,Astra Capital,Home & Kitchen,Astra,198.00,3.97




ORDERS
Shape : (20000, 5)



...

,order_item_id,order_id,product_id,user_id,quantity,item_price,item_total
0,I00000001,O00000001,P001758,U009310,2,8.07,16.14
1,I00000002,O00000001,P001119,U009310,1,74.08,74.08
2,I00000003,O00000001,P001794,U009310,1,576.97,576.97
3,I00000004,O00000001,P001038,U009310,1,22.47,22.47
4,I00000005,O00000002,P000859,U003247,1,422.22,422.22




EVENTS
Shape : (80000, 5)



,event_id,user_id,product_id,event_type,event_timestamp
0,E00000001,U009798,P001393,cart,2025-07-08T14:28:55.893919
1,E00000002,U005881,P000669,view,2025-10-19T23:00:44.067982
2,E00000003,U006348,P001404,view,2025-05-09T07:02:42.256662
3,E00000004,U002664,P000400,cart,2025-07-19T22:47:07.019634
4,E00000005,U005776,P000392,view,2024-10-24T10:20:33.602165




REVIEWS
Shape : (15000, 7)



,review_id,order_id,product_id,user_id,rating,review_text,review_date
0,R00000528,O00000237,P001326,U001094,2,Color was different from images.,2025-10-14T12:03:56.749446
1,R00005792,O00002627,P000329,U001858,4,Highly recommend this brand.,2024-10-09T08:04:50.171793
2,R00036604,O00016798,P001160,U008109,4,Highly recommend this brand.,2024-06-03T05:11:16.787214
3,R00040163,O00018414,P001427,U006835,5,Highly recommend this brand.,2024-02-12T06:41:50.215810
4,R00031127,O00014300,P001639,U007148,3,Item arrived damaged.,2025-01-20T05:32:09.398860


In [7]:
for name, df in datasets.items():
    print("=" * 70)
    print(name.upper())
    print("=" * 70)

    df.info()

    print("\n")

USERS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   user_id      10000 non-null  object
 1   name         10000 non-null  object
 2   email        10000 non-null  object
 3   gender       10000 non-null  object
 4   city         10000 non-null  object
 5   signup_date  10000 non-null  object
dtypes: object(6)
memory usage: 468.9+ KB


PRODUCTS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    2000 non-null   object 
 1   product_name  2000 non-null   object 
 2   category      2000 non-null   object 
 3   brand         2000 non-null   object 
 4   price         2000 non-null   float64
 5   rating        2000 non-null   float64
dtypes: float64(2), object(4)
memory usage: 93.9+ KB


O

In [8]:
for name, df in datasets.items():

    print("=" * 70)
    print(name.upper())
    print("=" * 70)

    missing = df.isnull().sum()

    print(missing[missing > 0])

    print()

USERS
Series([], dtype: int64)

PRODUCTS
Series([], dtype: int64)

ORDERS
Series([], dtype: int64)

ORDER ITEMS
Series([], dtype: int64)

EVENTS
Series([], dtype: int64)

REVIEWS
Series([], dtype: int64)



In [9]:
for name, df in datasets.items():

    print(f"{name} Duplicate Rows : {df.duplicated().sum()}")

Users Duplicate Rows : 0
Products Duplicate Rows : 0
Orders Duplicate Rows : 0
Order Items Duplicate Rows : 0
Events Duplicate Rows : 0
Reviews Duplicate Rows : 0


In [10]:
for name, df in datasets.items():

    print("=" * 70)
    print(name.upper())
    print("=" * 70)

    display(df.describe(include="all"))

USERS


,user_id,name,email,gender,city,signup_date
count,10000,10000,10000,10000,10000,10000
unique,10000,9340,10000,3,7755,684
top,U009984,Michael Brown,watkinscharles@example.net,Other,North Michael,2024-09-12
freq,1,6,1,3419,13,27


PRODUCTS


,product_id,product_name,category,brand,price,rating
count,2000,2000,2000,2000,2000.00,2000.00
unique,2000,1848,10,12,NaN,NaN
top,P001984,Zenith Might,Clothing,Zenith,NaN,NaN
freq,1,3,213,190,NaN,NaN
mean,NaN,NaN,NaN,NaN,200.34,3.68
std,NaN,NaN,NaN,NaN,302.74,0.69
min,NaN,NaN,NaN,NaN,1.11,2.50
25%,NaN,NaN,NaN,NaN,36.34,3.07
50%,NaN,NaN,NaN,NaN,84.22,3.69
75%,NaN,NaN,NaN,NaN,210.82,4.25


ORDERS


...

,order_item_id,order_id,product_id,user_id,quantity,item_price,item_total
count,43525,43525,43525,43525,43525.00,43525.00,43525.00
unique,43525,20000,2000,8635,NaN,NaN,NaN
top,I00043525,O00012206,P000519,U003319,NaN,NaN,NaN
freq,1,6,38,24,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,1.40,196.64,273.84
std,NaN,NaN,NaN,NaN,0.66,297.26,471.59
min,NaN,NaN,NaN,NaN,1.00,1.11,1.11
25%,NaN,NaN,NaN,NaN,1.00,36.40,44.59
50%,NaN,NaN,NaN,NaN,1.00,83.17,105.34
75%,NaN,NaN,NaN,NaN,2.00,208.05,274.98


EVENTS


,event_id,user_id,product_id,event_type,event_timestamp
count,80000,80000,80000,80000,80000
unique,80000,9995,2000,4,80000
top,E00079984,U006046,P001112,view,2024-05-14T20:17:41.446172
freq,1,23,63,56013,1


REVIEWS


,review_id,order_id,product_id,user_id,rating,review_text,review_date
count,15000,15000,15000,15000,15000.00,15000,15000
unique,15000,11070,1999,6660,NaN,10,15000
top,R00023858,O00012017,P000497,U009903,NaN,Item arrived damaged.,2024-07-05T16:32:13.463919
freq,1,6,20,11,NaN,1550,1
mean,NaN,NaN,NaN,NaN,3.54,NaN,NaN
std,NaN,NaN,NaN,NaN,1.10,NaN,NaN
min,NaN,NaN,NaN,NaN,1.00,NaN,NaN
25%,NaN,NaN,NaN,NaN,3.00,NaN,NaN
50%,NaN,NaN,NaN,NaN,4.00,NaN,NaN
75%,NaN,NaN,NaN,NaN,4.00,NaN,NaN


In [11]:
users_df = users.copy()
products_df = products.copy()
orders_df = orders.copy()
order_items_df = order_items.copy()
events_df = events.copy()
reviews_df = reviews.copy()

print("✅ Working copies created successfully!")

✅ Working copies created successfully!


In [12]:
print("Users:", users_df.shape)
print("Products:", products_df.shape)
print("Orders:", orders_df.shape)
print("Order Items:", order_items_df.shape)
print("Events:", events_df.shape)
print("Reviews:", reviews_df.shape)

Users: (10000, 6)
Products: (2000, 6)
Orders: (20000, 5)
Order Items: (43525, 7)
Events: (80000, 5)
Reviews: (15000, 7)


In [13]:
for name, df in {
    "Users": users_df,
    "Products": products_df,
    "Orders": orders_df,
    "Order Items": order_items_df,
    "Events": events_df,
    "Reviews": reviews_df
}.items():

    print("="*60)
    print(name)
    print("="*60)

    display(df.isnull().sum().to_frame("Missing Values"))

Users


,Missing Values
user_id,0
name,0
email,0
gender,0
city,0
signup_date,0


Products


,Missing Values
product_id,0
product_name,0
category,0
brand,0
price,0
rating,0


Orders


...

,Missing Values
order_item_id,0
order_id,0
product_id,0
user_id,0
quantity,0
item_price,0
item_total,0


Events


,Missing Values
event_id,0
user_id,0
product_id,0
event_type,0
event_timestamp,0


Reviews


,Missing Values
review_id,0
order_id,0
product_id,0
user_id,0
rating,0
review_text,0
review_date,0


In [16]:
users_df["signup_date"] = pd.to_datetime(users_df["signup_date"])

orders_df["order_date"] = pd.to_datetime(orders_df["order_date"])

events_df["event_timestamp"] = pd.to_datetime(events_df["event_timestamp"])

reviews_df["review_date"] = pd.to_datetime(reviews_df["review_date"])

print("✅ Date columns converted successfully!")

✅ Date columns converted successfully!


In [17]:
print(users_df.dtypes)

print(products_df.dtypes)

print(orders_df.dtypes)

print(order_items_df.dtypes)

print(events_df.dtypes)

print(reviews_df.dtypes)

user_id                object
name                   object
email                  object
gender                 object
city                   object
signup_date    datetime64[ns]
dtype: object
product_id       object
product_name     object
category         object
brand            object
price           float64
rating          float64
dtype: object
order_id                object
user_id                 object
order_date      datetime64[ns]
order_status            object
total_amount           float64
dtype: object
order_item_id     object
order_id          object
product_id        object
user_id           object
quantity           int64
item_price       float64
item_total       float64
dtype: object
event_id                   object
user_id                    object
product_id                 object
event_type                 object
event_timestamp    datetime64[ns]
dtype: object
review_id              object
order_id               object
product_id             object
user_id         

In [18]:
summary = pd.DataFrame({
    "Dataset": [
        "Users",
        "Products",
        "Orders",
        "Order Items",
        "Events",
        "Reviews"
    ],
    "Rows": [
        users_df.shape[0],
        products_df.shape[0],
        orders_df.shape[0],
        order_items_df.shape[0],
        events_df.shape[0],
        reviews_df.shape[0]
    ],
    "Columns": [
        users_df.shape[1],
        products_df.shape[1],
        orders_df.shape[1],
        order_items_df.shape[1],
        events_df.shape[1],
        reviews_df.shape[1]
    ]
})

summary

,Dataset,Rows,Columns
0,Users,10000,6
1,Products,2000,6
2,Orders,20000,5
3,Order Items,43525,7
4,Events,80000,5
5,Reviews,15000,7


# Business KPIs

In [19]:
# Total Users
total_users = users_df["user_id"].nunique()

# Total Products
total_products = products_df["product_id"].nunique()

# Total Orders
total_orders = orders_df["order_id"].nunique()

# Total Events
total_events = events_df.shape[0]

# Total Reviews
total_reviews = reviews_df.shape[0]

print(f"👥 Total Users      : {total_users:,}")
print(f"📦 Total Products   : {total_products:,}")
print(f"🛒 Total Orders     : {total_orders:,}")
print(f"🖱️ Total Events      : {total_events:,}")
print(f"⭐ Total Reviews    : {total_reviews:,}")

👥 Total Users      : 10,000
📦 Total Products   : 2,000
🛒 Total Orders     : 20,000
🖱️ Total Events      : 80,000
⭐ Total Reviews    : 15,000


In [20]:
overview = pd.DataFrame({
    "Metric": [
        "Total Users",
        "Total Products",
        "Total Orders",
        "Total Events",
        "Total Reviews"
    ],
    "Value": [
        total_users,
        total_products,
        total_orders,
        total_events,
        total_reviews
    ]
})

overview

,Metric,Value
0,Total Users,10000
1,Total Products,2000
2,Total Orders,20000
3,Total Events,80000
4,Total Reviews,15000


In [21]:
signup_trend = (
    users_df
    .groupby(users_df["signup_date"].dt.to_period("M"))
    .size()
    .reset_index(name="New Users")
)

signup_trend["signup_date"] = signup_trend["signup_date"].astype(str)

import plotly.express as px

fig = px.line(
    signup_trend,
    x="signup_date",
    y="New Users",
    title="Monthly User Sign-ups"
)

fig.show()

In [22]:
gender = users_df["gender"].value_counts().reset_index()
gender.columns = ["Gender", "Users"]

fig = px.pie(
    gender,
    names="Gender",
    values="Users",
    title="Gender Distribution"
)

fig.show()

In [23]:
city = (
    users_df["city"]
    .value_counts()
    .head(10)
    .reset_index()
)

city.columns = ["City", "Users"]

fig = px.bar(
    city,
    x="City",
    y="Users",
    title="Top 10 Cities by Users"
)

fig.show()

# Revenue Analysis

In [24]:
# Merge Orders and Order Items
sales_df = pd.merge(
    orders_df,
    order_items_df,
    on="order_id",
    how="inner"
)

# Merge with Products
sales_df = pd.merge(
    sales_df,
    products_df,
    on="product_id",
    how="left"
)

print("Merged Dataset Shape:", sales_df.shape)

sales_df.head()

Merged Dataset Shape: (43525, 16)


,order_id,user_id_x,order_date,order_status,total_amount,order_item_id,product_id,user_id_y,quantity,item_price,item_total,product_name,category,brand,price,rating
0,O00000001,U009310,2025-09-09 14:52:37.292731,processing,689.66,I00000001,P001758,U009310,2,8.07,16.14,Everest Whole,Pet Supplies,Everest,8.07,2.69
1,O00000001,U009310,2025-09-09 14:52:37.292731,processing,689.66,I00000002,P001119,U009310,1,74.08,74.08,Nimbus Minute,Beauty,Nimbus,74.08,3.62
2,O00000001,U009310,2025-09-09 14:52:37.292731,processing,689.66,I00000003,P001794,U009310,1,576.97,576.97,Willow Treatment,Automotive,Willow,576.97,3.89
3,O00000001,U009310,2025-09-09 14:52:37.292731,processing,689.66,I00000004,P001038,U009310,1,22.47,22.47,Nimbus Deal,Books,Nimbus,22.47,4.85
4,O00000002,U003247,2025-04-15 01:18:27.193404,completed,1666.85,I00000005,P000859,U003247,1,422.22,422.22,Pulse Decide,Electronics,Pulse,422.22,3.51


In [25]:
print(sales_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43525 entries, 0 to 43524
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_id       43525 non-null  object        
 1   user_id_x      43525 non-null  object        
 2   order_date     43525 non-null  datetime64[ns]
 3   order_status   43525 non-null  object        
 4   total_amount   43525 non-null  float64       
 5   order_item_id  43525 non-null  object        
 6   product_id     43525 non-null  object        
 7   user_id_y      43525 non-null  object        
 8   quantity       43525 non-null  int64         
 9   item_price     43525 non-null  float64       
 10  item_total     43525 non-null  float64       
 11  product_name   43525 non-null  object        
 12  category       43525 non-null  object        
 13  brand          43525 non-null  object        
 14  price          43525 non-null  float64       
 15  rating         4352

In [26]:
sales_df.describe()

,order_date,total_amount,quantity,item_price,item_total,price,rating
count,43525,43525.00,43525.00,43525.00,43525.00,43525.00,43525.00
mean,2024-12-06 20:22:16.754050816,786.24,1.40,196.64,273.84,196.64,3.67
min,2024-01-01 00:14:17.631198,1.11,1.00,1.11,1.11,1.11,2.50
25%,2024-06-19 06:49:34.263237120,200.07,1.00,36.40,44.59,36.40,3.07
50%,2024-12-05 10:39:39.387798016,477.43,1.00,83.17,105.34,83.17,3.68
75%,2025-05-27 16:33:46.914690048,1053.72,2.00,208.05,274.98,208.05,4.25
max,2025-11-14 23:17:25.548939,7950.74,3.00,2338.13,7014.39,2338.13,4.90
std,NaN,884.37,0.66,297.26,471.59,297.26,0.69


In [27]:
sales_df["Revenue"] = (
    sales_df["quantity"] *
    sales_df["price"]
)

sales_df.head()

,order_id,user_id_x,order_date,order_status,total_amount,order_item_id,product_id,user_id_y,quantity,item_price,item_total,product_name,category,brand,price,rating,Revenue
0,O00000001,U009310,2025-09-09 14:52:37.292731,processing,689.66,I00000001,P001758,U009310,2,8.07,16.14,Everest Whole,Pet Supplies,Everest,8.07,2.69,16.14
1,O00000001,U009310,2025-09-09 14:52:37.292731,processing,689.66,I00000002,P001119,U009310,1,74.08,74.08,Nimbus Minute,Beauty,Nimbus,74.08,3.62,74.08
2,O00000001,U009310,2025-09-09 14:52:37.292731,processing,689.66,I00000003,P001794,U009310,1,576.97,576.97,Willow Treatment,Automotive,Willow,576.97,3.89,576.97
3,O00000001,U009310,2025-09-09 14:52:37.292731,processing,689.66,I00000004,P001038,U009310,1,22.47,22.47,Nimbus Deal,Books,Nimbus,22.47,4.85,22.47
4,O00000002,U003247,2025-04-15 01:18:27.193404,completed,1666.85,I00000005,P000859,U003247,1,422.22,422.22,Pulse Decide,Electronics,Pulse,422.22,3.51,422.22


In [28]:
total_revenue = sales_df["Revenue"].sum()

print(f"💰 Total Revenue : ${total_revenue:,.2f}")

💰 Total Revenue : $11,918,668.95


In [29]:
total_quantity = sales_df["quantity"].sum()

print(f"📦 Total Quantity Sold : {total_quantity:,}")

📦 Total Quantity Sold : 60,818


In [30]:
average_order_value = (
    sales_df.groupby("order_id")["Revenue"]
            .sum()
            .mean()
)

print(f"🛒 Average Order Value : ${average_order_value:,.2f}")

🛒 Average Order Value : $595.93


In [31]:
products_df.columns.tolist()

['product_id', 'product_name', 'category', 'brand', 'price', 'rating']

In [32]:
category_revenue = (
    sales_df
    .groupby("category")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

category_revenue.head(10)

,category,Revenue
0,Electronics,4961736.60
1,Automotive,2501360.55
2,Home & Kitchen,1132697.25
3,Sports,952403.23
4,Clothing,710953.62
5,Beauty,555775.00
6,Toys,385482.77
7,Pet Supplies,354035.82
8,Books,274825.95
9,Groceries,89398.16


In [33]:
fig = px.bar(
    category_revenue,
    x="category",
    y="Revenue",
    title="Revenue by Product Category",
    color="Revenue"
)

fig.show()

# Product Analytics

In [35]:
top_products = (
    sales_df.groupby("product_name")
    .agg(
        Quantity_Sold=("quantity", "sum"),
        Revenue=("Revenue", "sum")
    )
    .sort_values(by="Quantity_Sold", ascending=False)
    .head(10)
    .reset_index()
)

top_products

,product_name,Quantity_Sold,Revenue
0,Solace Particular,104,9094.84
1,Orion Coach,101,56742.57
2,Everest Beautiful,99,18007.77
3,Pulse Money,98,45777.03
4,Nimbus Word,97,46858.21
5,Harbor Much,96,12849.10
6,GreenLeaf Save,91,6695.72
7,Nimbus Seven,88,1896.88
8,Zenith Might,88,11868.28
9,Everest Response,88,19678.56


In [36]:
fig = px.bar(
    top_products,
    x="product_name",
    y="Quantity_Sold",
    title="Top 10 Best-Selling Products",
    color="Quantity_Sold",
    text="Quantity_Sold"
)

fig.update_layout(xaxis_title="Product")
fig.show()

In [37]:
top_revenue_products = (
    sales_df.groupby("product_name")
    .agg(
        Revenue=("Revenue", "sum")
    )
    .sort_values(by="Revenue", ascending=False)
    .head(10)
    .reset_index()
)

top_revenue_products

,product_name,Revenue
0,Willow Result,73525.32
1,Astra Pull,66814.40
2,Willow Special,62032.60
3,Acme Room,61307.64
4,Orion Group,60784.10
5,Orion Coach,56742.57
6,Zenith Phone,56419.48
7,Willow Hospital,56311.92
8,Nimbus Family,54411.20
9,Zenith Their,54399.32


In [38]:
fig = px.bar(
    top_revenue_products,
    x="product_name",
    y="Revenue",
    title="Top 10 Revenue Generating Products",
    color="Revenue",
    text_auto=".2s"
)

fig.update_layout(xaxis_title="Product")
fig.show()

In [39]:
products_df.columns.tolist()

['product_id', 'product_name', 'category', 'brand', 'price', 'rating']

In [40]:
category_summary = (
    sales_df.groupby("category")
    .agg(
        Revenue=("Revenue", "sum"),
        Quantity=("quantity", "sum"),
        Orders=("order_id", "nunique")
    )
    .sort_values(by="Revenue", ascending=False)
    .reset_index()
)

category_summary

,category,Revenue,Quantity,Orders
0,Electronics,4961736.60,5879,3859
1,Automotive,2501360.55,5921,3883
2,Home & Kitchen,1132697.25,5902,3874
3,Sports,952403.23,5656,3686
4,Clothing,710953.62,6410,4152
5,Beauty,555775.00,6329,4139
6,Toys,385482.77,6519,4250
7,Pet Supplies,354035.82,6717,4286
8,Books,274825.95,6068,3909
9,Groceries,89398.16,5417,3552


In [41]:
fig = px.bar(
    category_summary,
    x="category",
    y="Revenue",
    title="Revenue by Product Category",
    color="Revenue",
    text_auto=".2s"
)

fig.show()

In [42]:
fig = px.histogram(
    products_df,
    x="price",
    nbins=30,
    title="Product Price Distribution"
)

fig.show()

In [43]:
top_rated = (
    reviews_df.groupby("product_id")
    .agg(
        Average_Rating=("rating", "mean"),
        Total_Reviews=("rating", "count")
    )
    .query("Total_Reviews >= 5")
    .sort_values(by="Average_Rating", ascending=False)
    .head(10)
    .reset_index()
)

top_rated

,product_id,Average_Rating,Total_Reviews
0,P001990,4.62,8
1,P001042,4.60,5
2,P001045,4.57,7
3,P001877,4.50,6
4,P001608,4.50,6
5,P000447,4.50,6
6,P000618,4.50,8
7,P000522,4.50,6
8,P001500,4.43,7
9,P000863,4.43,7


In [44]:
sales_df.columns.tolist()

['order_id',
 'user_id_x',
 'order_date',
 'order_status',
 'total_amount',
 'order_item_id',
 'product_id',
 'user_id_y',
 'quantity',
 'item_price',
 'item_total',
 'product_name',
 'category',
 'brand',
 'price',
 'rating',
 'Revenue']

In [45]:
brand_revenue = (
    sales_df.groupby("brand")
    .agg(
        Revenue=("Revenue", "sum"),
        Products_Sold=("quantity", "sum")
    )
    .sort_values(by="Revenue", ascending=False)
    .head(10)
    .reset_index()
)

brand_revenue

,brand,Revenue,Products_Sold
0,Willow,1333667.54,5129
1,Orion,1156616.67,5034
2,Nimbus,1091999.73,5418
3,Acme,1043174.92,4751
4,Astra,1023491.29,4969
5,Zenith,1013086.63,5873
6,GreenLeaf,989095.30,5373
7,Harbor,921264.61,5357
8,Solace,891005.52,4791
9,Pulse,882109.91,4343


In [46]:
fig = px.bar(
    brand_revenue,
    x="brand",
    y="Revenue",
    title="Top 10 Brands by Revenue",
    color="Revenue",
    text_auto=".2s"
)

fig.show()

In [47]:
monthly_revenue = (
    sales_df.groupby(
        sales_df["order_date"].dt.to_period("M")
    )
    .agg(
        Revenue=("Revenue", "sum")
    )
    .reset_index()
)

monthly_revenue["order_date"] = monthly_revenue["order_date"].astype(str)

monthly_revenue

,order_date,Revenue
0,2024-01,543120.43
1,2024-02,513712.79
2,2024-03,541213.26
3,2024-04,545624.46
4,2024-05,552700.82
5,2024-06,548534.32
6,2024-07,587135.76
7,2024-08,519474.77
8,2024-09,529067.12
9,2024-10,529215.48


In [48]:
fig = px.line(
    monthly_revenue,
    x="order_date",
    y="Revenue",
    markers=True,
    title="Monthly Revenue Trend"
)

fig.show()

In [49]:
order_status = (
    sales_df["order_status"]
    .value_counts()
    .reset_index()
)

order_status.columns = ["Status", "Orders"]

order_status

,Status,Orders
0,shipped,8939
1,returned,8802
2,completed,8733
3,cancelled,8593
4,processing,8458


In [50]:
fig = px.pie(
    order_status,
    names="Status",
    values="Orders",
    title="Order Status Distribution"
)

fig.show()

In [51]:
top_customers = (
    sales_df.groupby("user_id_x")
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("order_id", "nunique")
    )
    .sort_values(by="Revenue", ascending=False)
    .head(10)
    .reset_index()
)

top_customers

,user_id_x,Revenue,Orders
0,U009931,13288.88,4
1,U006233,11234.22,3
2,U006469,10953.18,4
3,U008370,10190.11,2
4,U005702,10175.81,6
5,U009903,10118.31,5
6,U000006,9952.28,6
7,U004906,9538.69,5
8,U006231,9486.29,2
9,U007930,8947.98,9


In [52]:
fig = px.bar(
    top_customers,
    x="user_id_x",
    y="Revenue",
    title="Top 10 Customers by Revenue",
    color="Revenue",
    text_auto=".2s"
)

fig.show()

In [53]:
category_summary = (
    sales_df.groupby("category")
    .agg(
        Revenue=("Revenue", "sum"),
        Products_Sold=("quantity", "sum"),
        Orders=("order_id", "nunique")
    )
    .sort_values(by="Revenue", ascending=False)
    .reset_index()
)

category_summary

,category,Revenue,Products_Sold,Orders
0,Electronics,4961736.60,5879,3859
1,Automotive,2501360.55,5921,3883
2,Home & Kitchen,1132697.25,5902,3874
3,Sports,952403.23,5656,3686
4,Clothing,710953.62,6410,4152
5,Beauty,555775.00,6329,4139
6,Toys,385482.77,6519,4250
7,Pet Supplies,354035.82,6717,4286
8,Books,274825.95,6068,3909
9,Groceries,89398.16,5417,3552


In [54]:
fig = px.bar(
    category_summary,
    x="category",
    y="Revenue",
    color="Revenue",
    title="Revenue by Category",
    text_auto=".2s"
)

fig.show()

In [55]:
events_df["event_type"].value_counts()

,count
event_type,
view,56013
cart,12035
wishlist,7946
purchase,4006


In [56]:
event_counts = events_df["event_type"].value_counts().reset_index()
event_counts.columns = ["Event", "Count"]

fig = px.bar(
    event_counts,
    x="Event",
    y="Count",
    title="User Events Distribution",
    color="Count",
    text_auto=True
)

fig.show()

In [57]:
daily_events = (
    events_df
    .groupby(events_df["event_timestamp"].dt.date)
    .size()
    .reset_index(name="Events")
)

fig = px.line(
    daily_events,
    x="event_timestamp",
    y="Events",
    title="Daily User Activity"
)

fig.show()

In [58]:
views = events_df[events_df["event_type"] == "view"]

top_views = (
    views.groupby("product_id")
    .size()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name="Views")
)

top_views

,product_id,Views
0,P000516,46
1,P001942,42
2,P001832,42
3,P000608,42
4,P001193,42
5,P000101,42
6,P001771,42
7,P001435,42
8,P001932,41
9,P000463,41


In [59]:
purchases = events_df[events_df["event_type"] == "purchase"]

top_purchase = (
    purchases.groupby("product_id")
    .size()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name="Purchases")
)

top_purchase

,product_id,Purchases
0,P001113,8
1,P000984,8
2,P001235,7
3,P000454,7
4,P000510,7
5,P001011,7
6,P000258,7
7,P001356,6
8,P001175,6
9,P001575,6


In [60]:
# Event type distribution
events_df["event_type"].value_counts()

,count
event_type,
view,56013
cart,12035
wishlist,7946
purchase,4006


In [61]:
import plotly.express as px

event_counts = (
    events_df["event_type"]
    .value_counts()
    .reset_index()
)

event_counts.columns = ["Event Type", "Count"]

fig = px.bar(
    event_counts,
    x="Event Type",
    y="Count",
    title="Distribution of User Events",
    color="Count",
    text="Count"
)

fig.show()

In [62]:
daily_events = (
    events_df.groupby(events_df["event_timestamp"].dt.date)
    .size()
    .reset_index(name="Total Events")
)

fig = px.line(
    daily_events,
    x="event_timestamp",
    y="Total Events",
    title="Daily User Activity"
)

fig.show()

In [63]:
top_users = (
    events_df.groupby("user_id")
    .size()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name="Total Events")
)

top_users

,user_id,Total Events
0,U006046,23
1,U003116,19
2,U007233,19
3,U001901,18
4,U003023,18
5,U006050,18
6,U008485,18
7,U008798,18
8,U009022,18
9,U008873,18


In [64]:
fig = px.bar(
    top_users,
    x="user_id",
    y="Total Events",
    title="Top 10 Most Active Users",
    color="Total Events",
    text="Total Events"
)

fig.show()

In [65]:
views = events_df[events_df["event_type"] == "view"]

top_views = (
    views.groupby("product_id")
    .size()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name="Views")
)

top_views

,product_id,Views
0,P000516,46
1,P001942,42
2,P001832,42
3,P000608,42
4,P001193,42
5,P000101,42
6,P001771,42
7,P001435,42
8,P001932,41
9,P000463,41


In [66]:
fig = px.bar(
    top_views,
    x="product_id",
    y="Views",
    title="Top 10 Most Viewed Products",
    color="Views",
    text="Views"
)

fig.show()

In [67]:
purchases = events_df[events_df["event_type"] == "purchase"]

top_purchases = (
    purchases.groupby("product_id")
    .size()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name="Purchases")
)

top_purchases

,product_id,Purchases
0,P001113,8
1,P000984,8
2,P001235,7
3,P000454,7
4,P000510,7
5,P001011,7
6,P000258,7
7,P001356,6
8,P001175,6
9,P001575,6


In [68]:
fig = px.bar(
    top_purchases,
    x="product_id",
    y="Purchases",
    title="Top 10 Most Purchased Products",
    color="Purchases",
    text="Purchases"
)

fig.show()

# Funnel Analysis

In [69]:
# Count events in each stage
view_count = len(events_df[events_df["event_type"] == "view"])
cart_count = len(events_df[events_df["event_type"] == "cart"])
purchase_count = len(events_df[events_df["event_type"] == "purchase"])

print(f"Views      : {view_count:,}")
print(f"Cart       : {cart_count:,}")
print(f"Purchases  : {purchase_count:,}")

Views      : 56,013
Cart       : 12,035
Purchases  : 4,006


In [70]:
funnel_df = pd.DataFrame({
    "Stage": ["View", "Cart", "Purchase"],
    "Users": [view_count, cart_count, purchase_count]
})

funnel_df

,Stage,Users
0,View,56013
1,Cart,12035
2,Purchase,4006


In [71]:
fig = px.funnel(
    funnel_df,
    x="Users",
    y="Stage",
    title="User Conversion Funnel"
)

fig.show()

In [72]:
view_to_cart = (cart_count / view_count) * 100
cart_to_purchase = (purchase_count / cart_count) * 100
overall_conversion = (purchase_count / view_count) * 100

print(f"View → Cart Conversion      : {view_to_cart:.2f}%")
print(f"Cart → Purchase Conversion  : {cart_to_purchase:.2f}%")
print(f"Overall Conversion          : {overall_conversion:.2f}%")

View → Cart Conversion      : 21.49%
Cart → Purchase Conversion  : 33.29%
Overall Conversion          : 7.15%


In [73]:
dropoff_view = 100 - view_to_cart
dropoff_cart = 100 - cart_to_purchase

dropoff_df = pd.DataFrame({
    "Stage": ["View", "Cart"],
    "Drop-off (%)": [dropoff_view, dropoff_cart]
})

dropoff_df

,Stage,Drop-off (%)
0,View,78.51
1,Cart,66.71


In [74]:
fig = px.bar(
    dropoff_df,
    x="Stage",
    y="Drop-off (%)",
    color="Drop-off (%)",
    text_auto=".2f",
    title="Drop-off Percentage at Each Funnel Stage"
)

fig.show()

## Funnel Insights

- Most users begin by viewing products.
- A portion of users add products to their cart.
- Some users complete a purchase.
- The largest drop-off stage represents the biggest opportunity for improving conversion.
- Product, UX, and marketing teams can use these insights to optimize the customer journey.

# Customer Segmentation

In [76]:
customer_summary = (
    sales_df.groupby("user_id_x")
    .agg(
        Total_Revenue=("Revenue", "sum"),
        Total_Orders=("order_id", "nunique"),
        Total_Items=("quantity", "sum")
    )
    .reset_index()
)

customer_summary.head()

,user_id_x,Total_Revenue,Total_Orders,Total_Items
0,U000001,1392.89,1,8
1,U000002,1808.15,4,14
2,U000003,1153.01,3,6
3,U000004,7620.84,3,19
4,U000005,12.66,1,1


In [77]:
top_customers = (
    customer_summary
    .sort_values(by="Total_Revenue", ascending=False)
    .head(10)
)

top_customers

,user_id_x,Total_Revenue,Total_Orders,Total_Items
8571,U009931,13288.88,4,16
5374,U006233,11234.22,3,20
5572,U006469,10953.18,4,28
7222,U008370,10190.11,2,13
4918,U005702,10175.81,6,25
8549,U009903,10118.31,5,26
5,U000006,9952.28,6,19
4238,U004906,9538.69,5,26
5372,U006231,9486.29,2,14
6834,U007930,8947.98,9,25


In [78]:
fig = px.bar(
    top_customers,
    x="user_id_x",
    y="Total_Revenue",
    color="Total_Revenue",
    text_auto=".2s",
    title="Top 10 Customers by Revenue"
)

fig.show()

In [79]:
customer_summary["Customer_Segment"] = pd.qcut(
    customer_summary["Total_Revenue"],
    q=4,
    labels=[
        "Bronze",
        "Silver",
        "Gold",
        "Platinum"
    ]
)

customer_summary.head()

,user_id_x,Total_Revenue,Total_Orders,Total_Items,Customer_Segment
0,U000001,1392.89,1,8,Gold
1,U000002,1808.15,4,14,Gold
2,U000003,1153.01,3,6,Gold
3,U000004,7620.84,3,19,Platinum
4,U000005,12.66,1,1,Bronze


In [80]:
segment_summary = (
    customer_summary["Customer_Segment"]
    .value_counts()
    .reset_index()
)

segment_summary.columns = [
    "Segment",
    "Customers"
]

segment_summary

,Segment,Customers
0,Bronze,2159
1,Silver,2159
2,Platinum,2159
3,Gold,2158


In [81]:
fig = px.pie(
    segment_summary,
    names="Segment",
    values="Customers",
    title="Customer Segments"
)

fig.show()

In [82]:
segment_revenue = (
    customer_summary.groupby("Customer_Segment")
    .agg(
        Average_Revenue=("Total_Revenue", "mean"),
        Average_Orders=("Total_Orders", "mean")
    )
    .reset_index()
)

segment_revenue

,Customer_Segment,Average_Revenue,Average_Orders
0,Bronze,168.79,1.38
1,Silver,621.53,2.01
2,Gold,1381.96,2.54
3,Platinum,3348.82,3.34


In [83]:
fig = px.bar(
    segment_revenue,
    x="Customer_Segment",
    y="Average_Revenue",
    color="Average_Revenue",
    text_auto=".2s",
    title="Average Revenue by Customer Segment"
)

fig.show()

## Customer Segmentation Insights

- Platinum customers contribute the highest revenue.
- Bronze customers represent opportunities for targeted marketing campaigns.
- Gold and Platinum customers should be prioritized for loyalty and retention programs.
- Customer segmentation enables personalized business strategies.

# Cohort & Retention Analysis

In [84]:
# Keep only completed orders
completed_orders = sales_df[
    sales_df["order_status"] == "Completed"
].copy()

# Purchase month
completed_orders["OrderMonth"] = (
    completed_orders["order_date"]
    .dt.to_period("M")
)

# First purchase month (cohort)
completed_orders["CohortMonth"] = (
    completed_orders
    .groupby("user_id_x")["OrderMonth"]
    .transform("min")
)

completed_orders.head()

,order_id,user_id_x,order_date,order_status,total_amount,order_item_id,product_id,user_id_y,quantity,item_price,item_total,product_name,category,brand,price,rating,Revenue,OrderMonth,CohortMonth


In [85]:
completed_orders["CohortIndex"] = (
    completed_orders["OrderMonth"].astype(int)
    - completed_orders["CohortMonth"].astype(int)
)

completed_orders.head()

,order_id,user_id_x,order_date,order_status,total_amount,order_item_id,product_id,user_id_y,quantity,item_price,item_total,product_name,category,brand,price,rating,Revenue,OrderMonth,CohortMonth,CohortIndex


In [86]:
cohort_data = (
    completed_orders
    .groupby(["CohortMonth", "CohortIndex"])
    .agg(Customers=("user_id_x", "nunique"))
    .reset_index()
)

cohort_data.head()

,CohortMonth,CohortIndex,Customers


In [87]:
cohort_table = cohort_data.pivot(
    index="CohortMonth",
    columns="CohortIndex",
    values="Customers"
)

cohort_table

CohortIndex
CohortMonth


In [88]:
retention_table = cohort_table.divide(
    cohort_table.iloc[:,0],
    axis=0
).round(3)

retention_table

IndexError: single positional indexer is out-of-bounds

In [89]:
print(completed_orders.shape)
completed_orders.head()


(0, 20)


,order_id,user_id_x,order_date,order_status,total_amount,order_item_id,product_id,user_id_y,quantity,item_price,item_total,product_name,category,brand,price,rating,Revenue,OrderMonth,CohortMonth,CohortIndex


In [90]:
sales_df["order_status"].value_counts()

,count
order_status,
shipped,8939
returned,8802
completed,8733
cancelled,8593
processing,8458


In [91]:
print(cohort_table.shape)
cohort_table

(0, 0)


CohortIndex
CohortMonth


In [92]:
sales_df["order_status"].value_counts()

,count
order_status,
shipped,8939
returned,8802
completed,8733
cancelled,8593
processing,8458


In [93]:
print(cohort_table.shape)

(0, 0)


In [94]:
completed_orders = sales_df[
    sales_df["order_status"] == "completed"
].copy()

print(completed_orders.shape)

(8733, 17)


In [95]:
completed_orders["OrderMonth"] = (
    completed_orders["order_date"].dt.to_period("M")
)

completed_orders.head()

,order_id,user_id_x,order_date,order_status,total_amount,order_item_id,product_id,user_id_y,quantity,item_price,item_total,product_name,category,brand,price,rating,Revenue,OrderMonth
4,O00000002,U003247,2025-04-15 01:18:27.193404,completed,1666.85,I00000005,P000859,U003247,1,422.22,422.22,Pulse Decide,Electronics,Pulse,422.22,3.51,422.22,2025-04
5,O00000002,U003247,2025-04-15 01:18:27.193404,completed,1666.85,I00000006,P000803,U003247,1,953.03,953.03,GreenLeaf Item,Electronics,GreenLeaf,953.03,3.43,953.03,2025-04
6,O00000002,U003247,2025-04-15 01:18:27.193404,completed,1666.85,I00000007,P000324,U003247,1,60.11,60.11,Harbor Visit,Toys,Harbor,60.11,3.20,60.11,2025-04
7,O00000002,U003247,2025-04-15 01:18:27.193404,completed,1666.85,I00000008,P000454,U003247,1,231.49,231.49,Solace Different,Home & Kitchen,Solace,231.49,3.58,231.49,2025-04
17,O00000006,U003449,2024-10-13 09:02:26.848944,completed,2258.34,I00000018,P000054,U003449,3,435.87,1307.61,GreenLeaf Hair,Automotive,GreenLeaf,435.87,4.04,1307.61,2024-10


In [96]:
completed_orders["CohortMonth"] = (
    completed_orders.groupby("user_id_x")["OrderMonth"]
    .transform("min")
)

completed_orders.head()

,order_id,user_id_x,order_date,order_status,total_amount,order_item_id,product_id,user_id_y,quantity,item_price,item_total,product_name,category,brand,price,rating,Revenue,OrderMonth,CohortMonth
4,O00000002,U003247,2025-04-15 01:18:27.193404,completed,1666.85,I00000005,P000859,U003247,1,422.22,422.22,Pulse Decide,Electronics,Pulse,422.22,3.51,422.22,2025-04,2024-06
5,O00000002,U003247,2025-04-15 01:18:27.193404,completed,1666.85,I00000006,P000803,U003247,1,953.03,953.03,GreenLeaf Item,Electronics,GreenLeaf,953.03,3.43,953.03,2025-04,2024-06
6,O00000002,U003247,2025-04-15 01:18:27.193404,completed,1666.85,I00000007,P000324,U003247,1,60.11,60.11,Harbor Visit,Toys,Harbor,60.11,3.20,60.11,2025-04,2024-06
7,O00000002,U003247,2025-04-15 01:18:27.193404,completed,1666.85,I00000008,P000454,U003247,1,231.49,231.49,Solace Different,Home & Kitchen,Solace,231.49,3.58,231.49,2025-04,2024-06
17,O00000006,U003449,2024-10-13 09:02:26.848944,completed,2258.34,I00000018,P000054,U003449,3,435.87,1307.61,GreenLeaf Hair,Automotive,GreenLeaf,435.87,4.04,1307.61,2024-10,2024-10


In [97]:
cohort_data = (
    completed_orders
    .groupby(["CohortMonth", "OrderMonth"])
    .agg(Customers=("user_id_x", "nunique"))
    .reset_index()
)

cohort_data.head()

,CohortMonth,OrderMonth,Customers
0,2024-01,2024-01,167
1,2024-01,2024-02,1
2,2024-01,2024-03,3
3,2024-01,2024-04,1
4,2024-01,2024-05,3


In [98]:
cohort_data.head()

,CohortMonth,OrderMonth,Customers
0,2024-01,2024-01,167
1,2024-01,2024-02,1
2,2024-01,2024-03,3
3,2024-01,2024-04,1
4,2024-01,2024-05,3


In [100]:
cohort_data["CohortIndex"] = (
    (cohort_data["OrderMonth"].dt.year - cohort_data["CohortMonth"].dt.year) * 12
    + (cohort_data["OrderMonth"].dt.month - cohort_data["CohortMonth"].dt.month)
)

cohort_data.head()

,CohortMonth,OrderMonth,Customers,CohortIndex
0,2024-01,2024-01,167,0
1,2024-01,2024-02,1,1
2,2024-01,2024-03,3,2
3,2024-01,2024-04,1,3
4,2024-01,2024-05,3,4


In [101]:
cohort_table = cohort_data.pivot(
    index="CohortMonth",
    columns="CohortIndex",
    values="Customers"
)

cohort_table

CohortIndex,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22
CohortMonth,,,,,,,,,,,,,,,,,,,,,,,
2024-01,167.00,1.00,3.00,1.00,3.00,4.00,3.00,1.00,3.00,3.00,NaN,3.00,NaN,3.00,NaN,1.00,1.00,5.00,2.00,5.00,NaN,1.00,1.00
2024-02,145.00,2.00,2.00,1.00,1.00,6.00,1.00,1.00,5.00,3.00,1.00,4.00,NaN,3.00,3.00,3.00,2.00,4.00,2.00,1.00,1.00,2.00,NaN
2024-03,207.00,6.00,6.00,3.00,6.00,3.00,2.00,6.00,4.00,11.00,7.00,4.00,6.00,4.00,4.00,3.00,3.00,NaN,NaN,2.00,1.00,NaN,NaN
2024-04,179.00,1.00,5.00,3.00,6.00,3.00,2.00,3.00,2.00,3.00,1.00,2.00,3.00,4.00,3.00,2.00,1.00,5.00,2.00,1.00,NaN,NaN,NaN
2024-05,156.00,7.00,2.00,5.00,3.00,2.00,5.00,4.00,3.00,2.00,3.00,3.00,2.00,5.00,3.00,7.00,1.00,1.00,2.00,NaN,NaN,NaN,NaN
2024-06,171.00,2.00,2.00,2.00,2.00,7.00,2.00,4.00,7.00,1.00,5.00,3.00,3.00,2.00,3.00,4.00,5.00,1.00,NaN,NaN,NaN,NaN,NaN
2024-07,163.00,4.00,5.00,1.00,4.00,4.00,4.00,NaN,4.00,3.00,3.00,3.00,6.00,1.00,2.00,2.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-08,167.00,1.00,2.00,3.00,5.00,4.00,4.00,1.00,2.00,2.00,4.00,3.00,3.00,5.00,3.00,1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-09,147.00,3.00,3.00,6.00,3.00,2.00,1.00,1.00,2.00,NaN,3.00,4.00,3.00,2.00,2.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [102]:
retention_table = cohort_table.div(
    cohort_table.iloc[:, 0],
    axis=0
).round(3)

retention_table

CohortIndex,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22
CohortMonth,,,,,,,,,,,,,,,,,,,,,,,
2024-01,1.00,0.01,0.02,0.01,0.02,0.02,0.02,0.01,0.02,0.02,NaN,0.02,NaN,0.02,NaN,0.01,0.01,0.03,0.01,0.03,NaN,0.01,0.01
2024-02,1.00,0.01,0.01,0.01,0.01,0.04,0.01,0.01,0.03,0.02,0.01,0.03,NaN,0.02,0.02,0.02,0.01,0.03,0.01,0.01,0.01,0.01,NaN
2024-03,1.00,0.03,0.03,0.01,0.03,0.01,0.01,0.03,0.02,0.05,0.03,0.02,0.03,0.02,0.02,0.01,0.01,NaN,NaN,0.01,0.01,NaN,NaN
2024-04,1.00,0.01,0.03,0.02,0.03,0.02,0.01,0.02,0.01,0.02,0.01,0.01,0.02,0.02,0.02,0.01,0.01,0.03,0.01,0.01,NaN,NaN,NaN
2024-05,1.00,0.04,0.01,0.03,0.02,0.01,0.03,0.03,0.02,0.01,0.02,0.02,0.01,0.03,0.02,0.04,0.01,0.01,0.01,NaN,NaN,NaN,NaN
2024-06,1.00,0.01,0.01,0.01,0.01,0.04,0.01,0.02,0.04,0.01,0.03,0.02,0.02,0.01,0.02,0.02,0.03,0.01,NaN,NaN,NaN,NaN,NaN
2024-07,1.00,0.03,0.03,0.01,0.03,0.03,0.03,NaN,0.03,0.02,0.02,0.02,0.04,0.01,0.01,0.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-08,1.00,0.01,0.01,0.02,0.03,0.02,0.02,0.01,0.01,0.01,0.02,0.02,0.02,0.03,0.02,0.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-09,1.00,0.02,0.02,0.04,0.02,0.01,0.01,0.01,0.01,NaN,0.02,0.03,0.02,0.01,0.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [103]:
import plotly.express as px

fig = px.imshow(
    retention_table,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Blues",
    title="Customer Retention Heatmap"
)

fig.show()

TypeError: Type is not JSON serializable: Period

In [104]:
retention_plot = retention_table.copy()

retention_plot.index = retention_plot.index.astype(str)
retention_plot.columns = retention_plot.columns.astype(str)

retention_plot.head()

CohortIndex,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22
CohortMonth,,,,,,,,,,,,,,,,,,,,,,,
2024-01,1.00,0.01,0.02,0.01,0.02,0.02,0.02,0.01,0.02,0.02,NaN,0.02,NaN,0.02,NaN,0.01,0.01,0.03,0.01,0.03,NaN,0.01,0.01
2024-02,1.00,0.01,0.01,0.01,0.01,0.04,0.01,0.01,0.03,0.02,0.01,0.03,NaN,0.02,0.02,0.02,0.01,0.03,0.01,0.01,0.01,0.01,NaN
2024-03,1.00,0.03,0.03,0.01,0.03,0.01,0.01,0.03,0.02,0.05,0.03,0.02,0.03,0.02,0.02,0.01,0.01,NaN,NaN,0.01,0.01,NaN,NaN
2024-04,1.00,0.01,0.03,0.02,0.03,0.02,0.01,0.02,0.01,0.02,0.01,0.01,0.02,0.02,0.02,0.01,0.01,0.03,0.01,0.01,NaN,NaN,NaN
2024-05,1.00,0.04,0.01,0.03,0.02,0.01,0.03,0.03,0.02,0.01,0.02,0.02,0.01,0.03,0.02,0.04,0.01,0.01,0.01,NaN,NaN,NaN,NaN


In [105]:
import plotly.express as px

fig = px.imshow(
    retention_plot,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Blues",
    title="Customer Retention Heatmap"
)

fig.show()

In [106]:
print(retention_table.dtypes)
print(retention_table.shape)
retention_table.head()

CohortIndex
0     float64
1     float64
2     float64
3     float64
4     float64
5     float64
6     float64
7     float64
8     float64
9     float64
10    float64
11    float64
12    float64
13    float64
14    float64
15    float64
16    float64
17    float64
18    float64
19    float64
20    float64
21    float64
22    float64
dtype: object
(23, 23)


CohortIndex,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22
CohortMonth,,,,,,,,,,,,,,,,,,,,,,,
2024-01,1.00,0.01,0.02,0.01,0.02,0.02,0.02,0.01,0.02,0.02,NaN,0.02,NaN,0.02,NaN,0.01,0.01,0.03,0.01,0.03,NaN,0.01,0.01
2024-02,1.00,0.01,0.01,0.01,0.01,0.04,0.01,0.01,0.03,0.02,0.01,0.03,NaN,0.02,0.02,0.02,0.01,0.03,0.01,0.01,0.01,0.01,NaN
2024-03,1.00,0.03,0.03,0.01,0.03,0.01,0.01,0.03,0.02,0.05,0.03,0.02,0.03,0.02,0.02,0.01,0.01,NaN,NaN,0.01,0.01,NaN,NaN
2024-04,1.00,0.01,0.03,0.02,0.03,0.02,0.01,0.02,0.01,0.02,0.01,0.01,0.02,0.02,0.02,0.01,0.01,0.03,0.01,0.01,NaN,NaN,NaN
2024-05,1.00,0.04,0.01,0.03,0.02,0.01,0.03,0.03,0.02,0.01,0.02,0.02,0.01,0.03,0.02,0.04,0.01,0.01,0.01,NaN,NaN,NaN,NaN


In [107]:
import plotly.graph_objects as go

fig = go.Figure(
    data=go.Heatmap(
        z=retention_table.fillna(0).values,
        x=retention_table.columns.astype(str),
        y=retention_table.index.astype(str),
        colorscale="Blues",
        text=retention_table.round(2).astype(str),
        texttemplate="%{text}",
        hoverongaps=False
    )
)

fig.update_layout(
    title="Customer Retention Heatmap",
    xaxis_title="Months Since First Purchase",
    yaxis_title="Cohort Month",
    width=1000,
    height=700
)

fig.show()

# A/B Testing Analysis

In [108]:
import numpy as np

# Ensure reproducibility
np.random.seed(42)

# Use unique customers
ab_test = customer_summary.copy()

# Randomly assign groups
ab_test["Variant"] = np.random.choice(
    ["A", "B"],
    size=len(ab_test),
    p=[0.5, 0.5]
)

ab_test.head()

,user_id_x,Total_Revenue,Total_Orders,Total_Items,Customer_Segment,Variant
0,U000001,1392.89,1,8,Gold,A
1,U000002,1808.15,4,14,Gold,B
2,U000003,1153.01,3,6,Gold,B
3,U000004,7620.84,3,19,Platinum,B
4,U000005,12.66,1,1,Bronze,A


In [109]:
# Customers with revenue > median are treated as "converted"
median_revenue = ab_test["Total_Revenue"].median()

ab_test["Converted"] = (
    ab_test["Total_Revenue"] > median_revenue
).astype(int)

ab_test.head()

,user_id_x,Total_Revenue,Total_Orders,Total_Items,Customer_Segment,Variant,Converted
0,U000001,1392.89,1,8,Gold,A,1
1,U000002,1808.15,4,14,Gold,B,1
2,U000003,1153.01,3,6,Gold,B,1
3,U000004,7620.84,3,19,Platinum,B,1
4,U000005,12.66,1,1,Bronze,A,0


In [110]:
conversion_summary = (
    ab_test.groupby("Variant")
    .agg(
        Customers=("Converted", "count"),
        Conversions=("Converted", "sum")
    )
)

conversion_summary["Conversion Rate"] = (
    conversion_summary["Conversions"] /
    conversion_summary["Customers"] * 100
)

conversion_summary

,Customers,Conversions,Conversion Rate
Variant,,,
A,4377,2155,49.23
B,4258,2162,50.78


In [111]:
import plotly.express as px

plot_df = conversion_summary.reset_index()

fig = px.bar(
    plot_df,
    x="Variant",
    y="Conversion Rate",
    color="Variant",
    text="Conversion Rate",
    title="Conversion Rate by Variant"
)

fig.update_traces(texttemplate="%{text:.2f}%")

fig.show()

In [112]:
from scipy.stats import chi2_contingency

contingency = [
    [
        conversion_summary.loc["A", "Conversions"],
        conversion_summary.loc["A", "Customers"] - conversion_summary.loc["A", "Conversions"]
    ],
    [
        conversion_summary.loc["B", "Conversions"],
        conversion_summary.loc["B", "Customers"] - conversion_summary.loc["B", "Conversions"]
    ]
]

chi2, p_value, dof, expected = chi2_contingency(contingency)

print(f"P-value : {p_value:.4f}")

P-value : 0.1586


In [113]:
if p_value < 0.05:
    print("✅ Statistically Significant Difference")
    print("Variant with the higher conversion rate should be considered.")
else:
    print("❌ No Statistically Significant Difference")
    print("There is not enough evidence to conclude one variant performs better.")

❌ No Statistically Significant Difference
There is not enough evidence to conclude one variant performs better.


## A/B Testing Insights

- Users were randomly assigned to Variant A and Variant B.
- Conversion was defined based on customer revenue relative to the median.
- A chi-square test was used to compare conversion rates.
- The results indicate whether the observed difference is statistically significant.
- These findings illustrate how controlled experiments can support product decisions.

In [114]:
import duckdb

duckdb.register("sales", sales_df)

In [115]:
duckdb.sql("""
SELECT
    ROUND(SUM(Revenue), 2) AS Total_Revenue
FROM sales;
""").df()

,Total_Revenue
0,11918668.95


In [116]:
duckdb.sql("""
SELECT
    product_name,
    SUM(quantity) AS Quantity_Sold
FROM sales
GROUP BY product_name
ORDER BY Quantity_Sold DESC
LIMIT 10;
""").df()

,product_name,Quantity_Sold
0,Solace Particular,104.00
1,Orion Coach,101.00
2,Everest Beautiful,99.00
3,Pulse Money,98.00
4,Nimbus Word,97.00
5,Harbor Much,96.00
6,GreenLeaf Save,91.00
7,Everest Response,88.00
8,Zenith Might,88.00
9,Nimbus Seven,88.00


In [117]:
duckdb.sql("""
SELECT
    category,
    ROUND(SUM(Revenue),2) AS Revenue
FROM sales
GROUP BY category
ORDER BY Revenue DESC;
""").df()

,category,Revenue
0,Electronics,4961736.60
1,Automotive,2501360.55
2,Home & Kitchen,1132697.25
3,Sports,952403.23
4,Clothing,710953.62
5,Beauty,555775.00
6,Toys,385482.77
7,Pet Supplies,354035.82
8,Books,274825.95
9,Groceries,89398.16


In [118]:
duckdb.sql("""
SELECT
    user_id_x,
    ROUND(SUM(Revenue),2) AS Revenue
FROM sales
GROUP BY user_id_x
ORDER BY Revenue DESC
LIMIT 10;
""").df()

,user_id_x,Revenue
0,U009931,13288.88
1,U006233,11234.22
2,U006469,10953.18
3,U008370,10190.11
4,U005702,10175.81
5,U009903,10118.31
6,U000006,9952.28
7,U004906,9538.69
8,U006231,9486.29
9,U007930,8947.98


In [119]:
duckdb.sql("""
SELECT
    strftime(order_date,'%Y-%m') AS Month,
    ROUND(SUM(Revenue),2) AS Revenue
FROM sales
GROUP BY Month
ORDER BY Month;
""").df()

,Month,Revenue
0,2024-01,543120.43
1,2024-02,513712.79
2,2024-03,541213.26
3,2024-04,545624.46
4,2024-05,552700.82
5,2024-06,548534.32
6,2024-07,587135.76
7,2024-08,519474.77
8,2024-09,529067.12
9,2024-10,529215.48


# Business Insights

### Revenue
- Category X generated the highest revenue.
- Revenue shows seasonal trends across months.

### Products
- A small group of products contributed a large share of sales.
- Several products generated high revenue despite lower sales volume.

### Customers
- Platinum customers contribute the largest share of revenue.
- Customer retention declines significantly after the first purchase.

### Funnel
- The largest drop-off occurs between product views and cart additions.
- Improving this stage could increase overall conversion.

### A/B Testing
- The experiment demonstrates how data-driven testing supports product decisions.

# Recommendations

1. Invest more in high-performing product categories.

2. Improve product pages for products with high views but low purchases.

3. Introduce loyalty programs for Platinum customers.

4. Improve checkout experience to reduce funnel drop-offs.

5. Promote highly rated products.

6. Continue A/B testing before releasing major product changes.

# Conclusion

This project analyzed a realistic e-commerce dataset containing users, products, orders, reviews, and user events.

The analysis combined Python, SQL, visualization, customer analytics, funnel analysis, cohort analysis, and A/B testing to generate business insights and recommendations.

The project demonstrates an end-to-end analytics workflow suitable for product and business decision-making.

UserFlow_Analytics.ipynb

UserFlow-Analytics/
│
├── UserFlow_Analytics.ipynb
├── README.md
├── requirements.txt
├── LICENSE
├── data/
│   └── sample datasets (or instructions to download)
├── images/
│   ├── dashboard.png
│   ├── funnel.png
│   ├── cohort.png
│   └── revenue.png

# Executive Summary

This analysis examined user behavior, sales performance, product performance, customer retention, and purchasing trends using an e-commerce dataset.

Key findings include:

- The business generated $X in total revenue.
- Category X contributed the highest revenue.
- The View → Cart stage had the highest funnel drop-off.
- Platinum customers generated the largest share of revenue.
- Customer retention decreased significantly after the first purchase.
- The A/B test demonstrated how experimentation can guide product decisions.

# Recommendations

1. Improve the View → Cart conversion rate through better product pages.

2. Launch loyalty programs for high-value customers.

3. Increase marketing spend on the highest-performing categories.

4. Improve low-rated products using customer feedback.

5. Continue experimenting with A/B tests before rolling out product changes.

6. Monitor customer retention monthly.

# Future Improvements

- Build a real-time dashboard using Streamlit.
- Connect to BigQuery instead of CSV files.
- Automate ETL pipelines.
- Add predictive models for customer churn.
- Forecast future revenue using machine learning.

# Conclusion

This project demonstrates a complete analytics workflow, including data cleaning, exploratory analysis, SQL, business KPIs, customer analytics, funnel analysis, cohort analysis, A/B testing, and executive reporting.

The insights generated can help product, marketing, and business teams make informed decisions.